# 02 · Forward pass and decoding

Load FlyNet (or CTRNN), run a looming stimulus through it and decode hit/miss and position/velocity.

**Requires** `flyvis` and `pytorch3d`.

In [ ]:
import torch
from flynet_robustness.models import get_flynet, get_stimulus_and_return_activity
from flynet_robustness.textures import get_texture

device = 'cuda' if torch.cuda.is_available() else 'cpu'
# 'flynet_synthesized' -> FlyNet, 'flynet_no_connectome' -> CTRNN
model1, model2, idxs1, idxs2 = get_flynet('flynet_synthesized', device=device)
print('decoded T4/T5 units:', len(idxs1))

In [ ]:
activity, speed, angle, position = get_stimulus_and_return_activity(
    model1, idxs1, model2, idxs2, hit=True, radius=4.0, shape='Sphere',
    texture=get_texture('Black'), time=1.0, device=device)
print('activity', tuple(activity.shape))  # (batch, time, n_units)

## Hit/miss probability
The hit/miss decoder is a frozen torch port of a scikit-learn MLP pipeline.

In [ ]:
from flynet_robustness.decoders import load_decoder
decoder = load_decoder(cnn=False, cnn_id='flynet_synthesized')
logits = decoder(activity.flatten(end_dim=1))
p_hit = torch.sigmoid(logits).detach().cpu().numpy()
import matplotlib.pyplot as plt
plt.plot(p_hit); plt.axhline(0.5, ls='--', c='gray'); plt.ylabel('p(hit)'); plt.xlabel('frame'); plt.show()

## Position / velocity
(requires trained regressors under `position_decoders/`, `velocity_decoders/`)

In [ ]:
from flynet_robustness.decoders import load_decoder_pos_vel
vel_decoder = load_decoder_pos_vel(load_velocity=True, device=device)
pred_vel = vel_decoder(activity.flatten(end_dim=1).to(device)).detach().cpu().numpy()
print('predicted velocity', pred_vel.shape)